# 13.5 - Self-Hosting with Ollama

**Module 13 - Cost & Performance** | Netsetos GenAI Engineering

This notebook turns the self-host-versus-API decision into seven small, runnable models: a break-even calculator, a which-model-fits check, a quantization VRAM sizer, a KV-cache context budgeter, a Modelfile parser with the one-line OpenAI wiring, a serving-knobs model, and a real total-cost-of-ownership calculator. Every block is keyless and illustrative - plain arithmetic you can re-run with your own numbers to see exactly where owning a GPU beats renting an API.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the one dependency the notebook touches - the OpenAI Python client, used only by the illustrative self-hosted-client cell (Step 5). Every other block is pure Python arithmetic with no external calls, so there is nothing to run here beyond making the client importable.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
!pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line environment-prep cell: it pip-installs `openai` (quietly) so the later illustrative client can point an OpenAI client at a local Ollama server. Nothing is imported or executed yet.

**How the code works - step by step**
- **`!pip install openai -q`** - installs the OpenAI SDK; the `-q` keeps the output quiet. Uncomment/keep it when running in Colab or a fresh environment.
- The trailing comment notes the lesson uses seeded, illustrative values throughout - the numbers are for intuition, not live measurements.

**In one line:** install the OpenAI client; everything else in this notebook is keyless arithmetic.

**What you'll see:** (no output - environment prep)

## 1 - The crossover: variable API cost vs fixed GPU cost

Self-hosting swaps a variable per-token bill for a fixed monthly GPU cost, and the two cross at a break-even volume: below it the API is cheaper, above it the GPU is. The catch is *which* API you compare to - a pricey frontier API breaks even at a moderate volume, a cheap open-model API at an enormous one. This block computes that crossover.

In [ ]:
# The crossover: an API bills PER TOKEN (variable), self-hosting is a FIXED monthly cost (the GPU).
# Below the break-even volume the API wins; above it, self-hosting wins. (Illustrative 2026 prices.)
GPU_FIXED = 1095.00          # a rented mid GPU: ~$1.50/hr x 730 hr/month (weights + serving)
API_FRONTIER = 10.00         # $/1M tokens on a frontier API (blended in+out)
API_OPEN = 0.40              # $/1M tokens on a cheap hosted OPEN-model API (Together-class, thin margins)

def breakeven(api_price):    # millions of tokens/month where fixed GPU == API bill
    return GPU_FIXED / api_price
print("Self-host fixed cost: ${:.0f}/month (the GPU is paid whether you use it or not).".format(GPU_FIXED))
print("Break-even volume vs each API (where the GPU cost equals the API bill):")
print("  vs a FRONTIER API (${:.2f}/1M):     {:>6.0f}M tokens/month".format(API_FRONTIER, breakeven(API_FRONTIER)))
print("  vs a cheap OPEN-model API (${:.2f}/1M): {:>6.0f}M tokens/month".format(API_OPEN, breakeven(API_OPEN)))
print()
for vol in [50, 300, 3000]:  # millions of tokens/month
    front, opn, self_host = vol * API_FRONTIER / 1, vol * API_OPEN / 1, GPU_FIXED
    winner = "self-host" if self_host < min(front, opn) else ("open-API" if opn < front else "frontier-API")
    print("  at {:>4}M tok/mo: frontier ${:>6.0f}, open-API ${:>5.0f}, self-host ${:>5.0f}  -> cheapest: {}".format(
        vol, front, opn, self_host, winner))
print()
print("Self-hosting beats a FRONTIER API at moderate volume, but rarely beats a cheap OPEN-model API,")
print("which already runs optimized infra at thin margins. WHICH API you compare to decides the crossover.")

# Output:
# Self-host fixed cost: $1095/month (the GPU is paid whether you use it or not).
# Break-even volume vs each API (where the GPU cost equals the API bill):
#   vs a FRONTIER API ($10.00/1M):        110M tokens/month
#   vs a cheap OPEN-model API ($0.40/1M):   2738M tokens/month
#
#   at   50M tok/mo: frontier $   500, open-API $   20, self-host $ 1095  -> cheapest: open-API
#   at  300M tok/mo: frontier $  3000, open-API $  120, self-host $ 1095  -> cheapest: open-API
#   at 3000M tok/mo: frontier $ 30000, open-API $ 1200, self-host $ 1095  -> cheapest: self-host
#
# Self-hosting beats a FRONTIER API at moderate volume, but rarely beats a cheap OPEN-model API,
# which already runs optimized infra at thin margins. WHICH API you compare to decides the crossover.

**Explanation**

A tiny cost model, not a model call: it divides a fixed GPU cost by two API prices to find the break-even token volume, then tabulates who wins at three sample volumes. The core idea is that a horizontal fixed-cost line crosses each sloped API line at a different point, and the cheap open-model line is almost never beaten.

**How the code works - step by step**
- **Constants** - `GPU_FIXED` ($1095/mo rented GPU), `API_FRONTIER` ($10/1M tokens), and `API_OPEN` ($0.40/1M on a thin-margin hosted open-model API).
- **`breakeven(api_price)`** - returns `GPU_FIXED / api_price`, the millions of tokens/month at which the fixed GPU cost equals the API bill.
- **The volume loop** - for 50M, 300M, and 3000M tokens/month it computes the frontier bill, the open-API bill, and the flat self-host cost, then names the cheapest of the three.

**In one line:** fixed GPU cost / API price = break-even volume - and the API you compare to moves it by orders of magnitude.

**What you'll see:** Break-even is ~110M tokens/month vs the frontier API but ~2738M vs the cheap open API. At 50M and 300M tok/mo the open API is cheapest; only at 3000M tok/mo does self-host ($1095) finally win.

## 2 - Which open models fit your GPU

Ollama is llama.cpp plus a daemon, CLI, HTTP API, and model registry: `ollama pull` then `ollama run` gives you a local OpenAI-compatible endpoint in about thirty seconds - as long as the model fits in memory. This block checks a 2026 open-model catalog against a given VRAM budget and calls out the biggest one that fits.

In [ ]:
# Ollama = llama.cpp + a daemon + a CLI + an HTTP API + a model registry. One command pulls a model
# and gives you an OpenAI-compatible endpoint on localhost:11434. Which 2026 open models fit YOUR GPU?
GPU_VRAM = 24                # your GPU, in GB (illustrative)
catalog = [                  # (ollama tag, params_B, note) - Q4_K_M VRAM approx = params * 4.5/8 + ~15% overhead
    ("llama3.2:3b",          3,  "tiny, fast, on a laptop"),
    ("qwen2.5-coder:7b",     7,  "strong coding for its size"),
    ("phi4:14b",            14,  "small-but-capable reasoning"),
    ("gemma3:27b",          27,  "high quality, mid GPU"),
    ("qwen2.5-coder:32b",   32,  "~92% HumanEval, code specialist"),
    ("llama3.3:70b",        70,  "frontier-class open weights"),
]
def q4_vram(p): return round(p * 4.5 / 8 * 1.15, 1)   # Q4_K_M bytes-per-param ~4.5 bits + runtime overhead
print("Ollama catalog at Q4_K_M vs a {} GB GPU (pull with: ollama pull <tag>):".format(GPU_VRAM))
for tag, p, note in catalog:
    v = q4_vram(p)
    fits = "FITS" if v <= GPU_VRAM else "too big"
    print("  {:<20} {:>2}B  ~{:>4.1f} GB  [{:<7}]  {}".format(tag, p, v, fits, note))
print()
biggest = max((p for tag, p, note in catalog if q4_vram(p) <= GPU_VRAM), default=0)
print("Biggest model that fits: {}B at Q4_K_M. `ollama run <tag>` and you have a local OpenAI endpoint in ~30s.".format(biggest))
print("The registry has Llama, Qwen, Gemma, Mistral, DeepSeek, Phi - all open weights, all one pull away.")

# Output:
# Ollama catalog at Q4_K_M vs a 24 GB GPU (pull with: ollama pull <tag>):
#   llama3.2:3b           3B  ~ 1.9 GB  [FITS   ]  tiny, fast, on a laptop
#   qwen2.5-coder:7b      7B  ~ 4.5 GB  [FITS   ]  strong coding for its size
#   phi4:14b             14B  ~ 9.1 GB  [FITS   ]  small-but-capable reasoning
#   gemma3:27b           27B  ~17.5 GB  [FITS   ]  high quality, mid GPU
#   qwen2.5-coder:32b    32B  ~20.7 GB  [FITS   ]  ~92% HumanEval, code specialist
#   llama3.3:70b         70B  ~45.3 GB  [too big]  frontier-class open weights
#
# Biggest model that fits: 32B at Q4_K_M. `ollama run <tag>` and you have a local OpenAI endpoint in ~30s.
# The registry has Llama, Qwen, Gemma, Mistral, DeepSeek, Phi - all open weights, all one pull away.

**Explanation**

A fit-checker over a hard-coded catalog: it estimates each model's Q4_K_M VRAM footprint from its parameter count and flags whether it fits the GPU. The point is the one hard rule - a model must fit in VRAM or it spills to disk and crawls - so the useful answer is the largest model under the limit.

**How the code works - step by step**
- **`catalog`** - a list of `(ollama tag, params_B, note)` tuples spanning Llama, Qwen, Phi, and Gemma from 3B to 70B.
- **`q4_vram(p)`** - estimates VRAM as params x 4.5 bits / 8, plus ~15% runtime overhead (the Q4_K_M bytes-per-param rule of thumb).
- **The catalog loop** - prints each model's size and a `FITS`/`too big` flag against `GPU_VRAM` (24 GB).
- **`biggest`** - a max over the params of only the models that fit, giving the largest model you can actually pull.

**In one line:** estimate each model's Q4 VRAM, keep the ones under your GPU, pull the biggest.

**What you'll see:** Against a 24 GB GPU, everything up to `qwen2.5-coder:32b` (~20.7 GB) fits; `llama3.3:70b` (~45.3 GB) is too big. It reports the biggest fitting model as 32B at Q4_K_M.

## 3 - Quantization: fit a big model on a small GPU

A model's size is parameters times bits-per-weight, over eight. Quantization stores each weight in fewer bits, and Q4_K_M is the 2026 sweet spot - about 95% of full-precision quality at roughly a quarter of the VRAM. This block sizes an 8B model across quantization levels and shows what Q4 buys you.

In [ ]:
# Quantization shrinks the weights so a big model fits a small GPU. VRAM ~= params x bits / 8.
# Q4_K_M is the 2026 sweet spot: ~95% of full-precision quality at ~4x smaller. Below Q4, quality drops.
params_B = 8                 # an 8B model
levels = [("FP16 (full)", 16, 1.00), ("Q8_0", 8, 0.99), ("Q4_K_M", 4.5, 0.95), ("Q2_K", 2.6, 0.82)]
print("An {}B model at different quantization levels (VRAM ~= params x bits / 8, + ~15% overhead):".format(params_B))
for name, bits, quality in levels:
    vram = round(params_B * bits / 8 * 1.15, 1)
    print("  {:<12} {:>4} bits  ~{:>4.1f} GB  quality ~{:.0%}".format(name, bits, vram, quality))
print()
fp16 = params_B * 16 / 8 * 1.15
q4 = params_B * 4.5 / 8 * 1.15
print("Q4_K_M cuts VRAM {:.1f}x ({:.1f} GB -> {:.1f} GB) for only about a 5% quality drop - the default choice.".format(fp16 / q4, fp16, q4))
print("An 8B model in FP16 needs a 24 GB card; at Q4_K_M (~5 GB) it runs on an 8 GB laptop GPU.")
print("Go below Q4 (Q2, Q3) only when you must - reasoning and code degrade first and most.")

# Output:
# An 8B model at different quantization levels (VRAM ~= params x bits / 8, + ~15% overhead):
#   FP16 (full)    16 bits  ~18.4 GB  quality ~100%
#   Q8_0            8 bits  ~ 9.2 GB  quality ~99%
#   Q4_K_M        4.5 bits  ~ 5.2 GB  quality ~95%
#   Q2_K          2.6 bits  ~ 3.0 GB  quality ~82%
#
# Q4_K_M cuts VRAM 3.6x (18.4 GB -> 5.2 GB) for only about a 5% quality drop - the default choice.
# An 8B model in FP16 needs a 24 GB card; at Q4_K_M (~5 GB) it runs on an 8 GB laptop GPU.
# Go below Q4 (Q2, Q3) only when you must - reasoning and code degrade first and most.

**Explanation**

A sizing table that pairs each quantization level with a VRAM estimate and a rough quality retention. It makes the tradeoff concrete: Q4_K_M is a small, mostly-invisible quality loss for a large memory saving, and dropping below Q4 costs quality fast - reasoning and code first.

**How the code works - step by step**
- **`levels`** - `(name, bits, quality)` for FP16, Q8_0, Q4_K_M, and Q2_K, with quality as a fraction of FP16.
- **The levels loop** - computes `params_B x bits / 8 x 1.15` (overhead) for each and prints VRAM alongside the quality estimate.
- **`fp16` / `q4`** - the two anchor sizes; their ratio is the memory saving Q4_K_M delivers.
- **Closing prints** - state the ~3.6x cut for ~5% quality loss, and warn that below Q4 reasoning and code degrade first and most.

**In one line:** fewer bits per weight = smaller model; Q4_K_M is ~4x smaller for only ~5% quality lost.

**What you'll see:** The 8B model drops from ~18.4 GB (FP16) to ~5.2 GB at Q4_K_M - a 3.6x cut for ~5% quality - so it runs on an 8 GB laptop GPU. Q2_K (~3.0 GB) falls to ~82% quality.

## 4 - Context costs memory: the KV cache

Total VRAM is not just the weights - it is weights plus the KV cache, the per-token attention state that grows linearly with context length. So `num_ctx` is a real memory budget, not a free dial. This block computes the KV cost per token and shows how a long context can eat more VRAM than the weights themselves.

In [ ]:
# The context window costs VRAM too. Total = model weights + the KV cache, and the KV cache grows
# LINEARLY with context length. So your real memory budget is weights + KV, not just the weights.
# KV bytes/token = 2 (K and V) x n_layers x n_kv_heads x head_dim x 2 bytes (fp16). (Llama-3-8B GQA numbers.)
n_layers, n_kv_heads, head_dim = 32, 8, 128
weights_gb = 4.7            # 8B at Q4_K_M on disk (~5 GB, the Step 3 estimate)
kv_per_token = 2 * n_layers * n_kv_heads * head_dim * 2      # bytes per token
def kv_gb(ctx): return round(kv_per_token * ctx / 1e9, 2)
print("KV cache per token: {:,} bytes ({:.0f} KB). It is paid for EVERY token in the context window.".format(kv_per_token, kv_per_token / 1024))
print("Total VRAM = {} GB weights + KV cache:".format(weights_gb))
for ctx in [8192, 32768, 131072]:
    total = round(weights_gb + kv_gb(ctx), 1)
    print("  num_ctx {:>6}:  weights {} GB + KV {:>4.1f} GB = {:>4.1f} GB total".format(ctx, weights_gb, kv_gb(ctx), total))
print()
budget = 12                # a 12 GB GPU
max_kv = budget - weights_gb - 0.8    # leave headroom for activations
max_ctx = int(max_kv * 1e9 / kv_per_token)
print("On a {} GB GPU, the KV cache can use ~{:.1f} GB, so the max context is about {:,} tokens.".format(budget, max_kv, max_ctx // 1000 * 1000))
print("Set num_ctx no larger than you need: a long context you never use just burns VRAM you could spend on a bigger model.")

# Output:
# KV cache per token: 131,072 bytes (128 KB). It is paid for EVERY token in the context window.
# Total VRAM = 4.7 GB weights + KV cache:
#   num_ctx   8192:  weights 4.7 GB + KV  1.1 GB =  5.8 GB total
#   num_ctx  32768:  weights 4.7 GB + KV  4.3 GB =  9.0 GB total
#   num_ctx 131072:  weights 4.7 GB + KV 17.2 GB = 21.9 GB total
#
# On a 12 GB GPU, the KV cache can use ~6.5 GB, so the max context is about 49,000 tokens.
# Set num_ctx no larger than you need: a long context you never use just burns VRAM you could spend on a bigger model.

**Explanation**

A memory budgeter using real Llama-3-8B GQA dimensions: it computes bytes-per-token for the KV cache, adds it to the weights across three context lengths, then inverts the math to find the max context a given GPU can hold. The takeaway is that an oversized `num_ctx` steals VRAM you could spend on a bigger model or more concurrency.

**How the code works - step by step**
- **Architecture constants** - `n_layers`, `n_kv_heads`, `head_dim` from Llama-3-8B, plus `weights_gb` (4.7 GB at Q4).
- **`kv_per_token`** - `2 (K and V) x n_layers x n_kv_heads x head_dim x 2 bytes` = bytes per token.
- **`kv_gb(ctx)`** - scales that per-token cost to gigabytes for a given context length.
- **The context loop** - prints weights + KV = total for 8K, 32K, and 128K contexts.
- **`max_ctx`** - inverts the budget: on a 12 GB GPU, subtract weights and headroom, divide the remaining VRAM by the per-token cost to get the largest context that fits.

**In one line:** KV cache = fixed bytes/token x context length, added on top of the weights - so size `num_ctx` to your task.

**What you'll see:** KV cache is 131,072 bytes (128 KB) per token; at 128K context it adds 17.2 GB, more than the 4.7 GB of weights. On a 12 GB GPU the max context is about 49,000 tokens.

## 5 - Wire it in: the OpenAI-compatible API + Modelfile

Ollama speaks the OpenAI API, so moving an app to a local model is a one-line `base_url` change to localhost:11434/v1. A Modelfile (`FROM`, `SYSTEM`, `PARAMETER`) bakes a system prompt and params into a named, reusable model you build with `ollama create`. This block parses a Modelfile and shows the wiring deterministically - no live server needed.

In [ ]:
# Ollama speaks the OpenAI API, so wiring an app in is a ONE-LINE base_url change. A Modelfile bakes a
# system prompt + params into a named, reusable model. Here we PARSE a Modelfile and show the request
# SHAPE deterministically (no live server needed); the live call is the illustrative block below.
MODELFILE = """FROM qwen2.5-coder:32b
SYSTEM You are a terse senior code reviewer. Reply with a diff, no prose.
PARAMETER temperature 0.2
PARAMETER num_ctx 16384"""
directives = {}
for line in MODELFILE.strip().splitlines():
    key, _, val = line.partition(" ")
    directives.setdefault(key, []).append(val)
print("A Modelfile bakes config into a reusable model (build it: ollama create my-reviewer -f Modelfile):")
for key in ["FROM", "SYSTEM", "PARAMETER"]:
    print("  {:<10} {}".format(key, "  |  ".join(directives.get(key, []))))
print()
CLOUD = "https://api.openai.com/v1"
LOCAL = "http://localhost:11434/v1"
print("Wiring an existing app to the local model is a ONE-LINE change:")
print("  cloud:  base_url = {}".format(CLOUD))
print("  local:  base_url = {}   (api_key is just a placeholder)".format(LOCAL))
print("Everything downstream - streaming, tools, JSON mode - is the SAME OpenAI-compatible call.")
print("So porting an app from a hosted API to a self-hosted model is usually an afternoon, not a rewrite.")

# Output:
# A Modelfile bakes config into a reusable model (build it: ollama create my-reviewer -f Modelfile):
#   FROM       qwen2.5-coder:32b
#   SYSTEM     You are a terse senior code reviewer. Reply with a diff, no prose.
#   PARAMETER  temperature 0.2  |  num_ctx 16384
#
# Wiring an existing app to the local model is a ONE-LINE change:
#   cloud:  base_url = https://api.openai.com/v1
#   local:  base_url = http://localhost:11434/v1   (api_key is just a placeholder)
# Everything downstream - streaming, tools, JSON mode - is the SAME OpenAI-compatible call.
# So porting an app from a hosted API to a self-hosted model is usually an afternoon, not a rewrite.

**Explanation**

A string-parsing demo, not a network call: it splits a Modelfile into its directives and prints the one-line difference between the cloud and local `base_url`. The point is that porting is an afternoon, not a rewrite - the request shape is identical, only the endpoint moves.

**How the code works - step by step**
- **`MODELFILE`** - a sample Modelfile with `FROM`, a `SYSTEM` persona, and two `PARAMETER` lines (temperature, num_ctx).
- **The parse loop** - `partition(" ")` splits each line into directive and value, grouping repeated directives (like `PARAMETER`) into a list.
- **The directive printout** - shows `FROM`/`SYSTEM`/`PARAMETER` as they'd build into a named model via `ollama create`.
- **`CLOUD` vs `LOCAL`** - the two `base_url` values, making explicit that swapping them (with a placeholder api_key) is the only code change.

**In one line:** a Modelfile bakes config into a named model, and pointing `base_url` at localhost:11434/v1 is the whole port.

**What you'll see:** Prints the parsed Modelfile (FROM qwen2.5-coder:32b, a reviewer SYSTEM prompt, two PARAMETERs) and the cloud-vs-local base_url swap, noting that streaming, tools, and JSON mode are the same call.

## 6 - The self-hosted client (illustrative)

This is the live version of Step 5's wiring: the standard OpenAI Python client, pointed at a local Ollama server, streaming from a Modelfile-built model. It is illustrative - it needs a running Ollama and the model pulled - so it is not executed in the notebook.

> **Needs a running Ollama server** and the `my-reviewer` model built - shown for reference, not run here.

In [ ]:
# SELF-HOSTED CLIENT - an illustrative minimal example (the OpenAI client, pointed at a local Ollama).
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # the ONLY change vs a hosted API

def review(code: str):
    # streaming works exactly as it would against a hosted API - Ollama speaks the OpenAI protocol
    stream = client.chat.completions.create(
        model="my-reviewer",                 # a Modelfile-built model: ollama create my-reviewer -f Modelfile
        messages=[{"role": "user", "content": code}],
        stream=True,
    )
    for chunk in stream:
        yield chunk.choices[0].delta.content or ""     # the same token stream your UI already handles
# Everything else - tools, JSON mode, temperature - is the identical OpenAI call, now running on YOUR GPU.
# Output: (illustrative minimal example - needs a running Ollama + the model pulled; not run here.)

**Explanation**

A reference client, not a runnable cell: it constructs an `OpenAI` client whose only non-standard argument is the local `base_url`, then defines a streaming `review` generator. It demonstrates that self-hosting changes exactly one line - everything downstream is the identical OpenAI call, now on your own GPU.

**How the code works - step by step**
- **`client = OpenAI(base_url=..., api_key="ollama")`** - the only change versus a hosted API; the api_key is a placeholder Ollama ignores.
- **`review(code)`** - a generator that calls `chat.completions.create` with `stream=True` against the Modelfile-built `my-reviewer` model.
- **The `for chunk` loop** - yields `chunk.choices[0].delta.content` tokens, the same streaming shape a UI already handles.

**In one line:** the standard OpenAI streaming client, one line repointed at localhost, running on your GPU.

**What you'll see:** No output - an illustrative snippet that requires a running Ollama plus the pulled model; it is not executed in the notebook.

## 7 - Serve it: concurrency and the ceiling

Production serving turns on two knobs - `OLLAMA_NUM_PARALLEL` (concurrent requests) and `OLLAMA_KEEP_ALIVE` (how long the model stays in VRAM) - and watches one metric, queue depth. But Ollama serves one-at-a-time at its core, so for many concurrent users you graduate to vLLM. This block models the knobs and the throughput gap.

In [ ]:
# Serving in production: Ollama loads ONE model and serves a few requests at once. Two knobs matter.
# OLLAMA_NUM_PARALLEL = concurrent requests per model; OLLAMA_KEEP_ALIVE = how long the model stays in VRAM.
NUM_PARALLEL = 4            # concurrent slots (default 4, or 1 if VRAM is tight)
KEEP_ALIVE_MIN = 5         # keep the model loaded 5 min so a burst does not trigger a 4-6s reload
reload_penalty_s = 5
print("Ollama serving knobs:")
print("  OLLAMA_NUM_PARALLEL={}: up to {} requests run at once; the rest queue.".format(NUM_PARALLEL, NUM_PARALLEL))
print("  OLLAMA_KEEP_ALIVE={}m: model stays in VRAM; too low (30s) and every burst reloads (+{}s latency).".format(KEEP_ALIVE_MIN, reload_penalty_s))
print()
# Queue depth is the cardinal metric: past 2 x NUM_PARALLEL, p95 latency doubles.
for queue in [NUM_PARALLEL, 2 * NUM_PARALLEL, 4 * NUM_PARALLEL]:
    state = "healthy" if queue <= 2 * NUM_PARALLEL else "p95 doubling - add replicas / a load balancer"
    print("  queue depth {:>2}:  {}".format(queue, state))
print()
# The ceiling: Ollama is one-user-at-a-time (llama.cpp); vLLM's continuous batching serves many at once.
ollama_tps, vllm_tps = 484, 8033      # illustrative Llama-70B on a Blackwell GPU
print("Ollama vs vLLM on the same GPU (illustrative Llama-70B): {} vs {:,} tokens/s = {:.0f}x throughput.".format(ollama_tps, vllm_tps, vllm_tps / ollama_tps))
print("Ollama is the right tool for dev and low concurrency; for many concurrent users you graduate to vLLM (Lesson 13.6).")

# Output:
# Ollama serving knobs:
#   OLLAMA_NUM_PARALLEL=4: up to 4 requests run at once; the rest queue.
#   OLLAMA_KEEP_ALIVE=5m: model stays in VRAM; too low (30s) and every burst reloads (+5s latency).
#
#   queue depth  4:  healthy
#   queue depth  8:  healthy
#   queue depth 16:  p95 doubling - add replicas / a load balancer
#
# Ollama vs vLLM on the same GPU (illustrative Llama-70B): 484 vs 8,033 tokens/s = 17x throughput.
# Ollama is the right tool for dev and low concurrency; for many concurrent users you graduate to vLLM (Lesson 13.6).

**Explanation**

A serving-behavior model: it describes what each knob does, classifies queue depth as healthy or degrading, and contrasts Ollama's throughput with vLLM's. The core idea is that Ollama has a hard concurrency ceiling - past ~2x NUM_PARALLEL p95 doubles, and llama.cpp doesn't truly batch - so scale means adding replicas or moving to vLLM.

**How the code works - step by step**
- **`NUM_PARALLEL` / `KEEP_ALIVE_MIN`** - the two knobs, with a note that too-short keep-alive triggers a multi-second reload on every burst.
- **The queue loop** - marks queue depth `healthy` up to 2x NUM_PARALLEL and `p95 doubling - add replicas` beyond it.
- **`ollama_tps` vs `vllm_tps`** - illustrative Llama-70B throughput on the same GPU; their ratio is the ~17x gain vLLM's continuous batching delivers.

**In one line:** two knobs plus queue depth manage Ollama's serving, but a ~17x throughput ceiling sends heavy concurrency to vLLM.

**What you'll see:** Queue depth 4 and 8 read healthy; at 16 (4x NUM_PARALLEL) p95 doubles and it says add replicas. Ollama vs vLLM is 484 vs 8,033 tokens/s = 17x throughput, pointing to Lesson 13.6.

## 8 - The real decision: TCO and utilization

The naive crossover from Step 1 misses two costs that flip real decisions: ops labor (10-20 engineer-hours a month, so raw GPU is often only a third of the bill) and utilization (a GPU you pay for 24/7 but barely use is expensive per token). This block runs the real total-cost-of-ownership math at three utilization levels.

In [ ]:
# The REAL decision is TCO at YOUR utilization - not the sticker GPU price. Two costs the naive crossover
# misses: OPS labor, and low UTILIZATION (a GPU you pay for 24/7 but barely use is expensive per token).
GPU_FIXED = 1095.00                 # rented GPU / month
OPS_HOURS, OPS_RATE = 15, 100       # ~15 hrs/month of engineer time at $100/hr (patching, monitoring, on-call)
ops_cost = OPS_HOURS * OPS_RATE
total_monthly = GPU_FIXED + ops_cost
CAPACITY_M = 6500                   # tokens/month the GPU could serve if kept BUSY (illustrative, millions)
print("True monthly TCO: ${:.0f} GPU + ${:.0f} ops ({} hrs x ${}) = ${:.0f}/month.".format(GPU_FIXED, ops_cost, OPS_HOURS, OPS_RATE, total_monthly))
print("Raw GPU is only about {:.0%} of the real cost - ops labor is the rest, and it never sleeps.".format(GPU_FIXED / total_monthly))
print()
API_FRONTIER, API_OPEN = 10.00, 0.40   # $/1M on a frontier API and on a cheap open-model API (from Step 1)
print("Effective cost per 1M tokens depends entirely on UTILIZATION (actual volume vs the {:,}M capacity):".format(CAPACITY_M))
for util in [0.03, 0.20, 0.80]:
    vol = CAPACITY_M * util
    per_m = total_monthly / vol
    vs_front = "beats" if per_m < API_FRONTIER else "LOSES to"
    vs_open = "beats" if per_m < API_OPEN else "LOSES to"
    print("  util {:>3.0%} ({:>5.0f}M tok/mo):  ${:>6.2f}/1M  ->  {} the ${:.0f} frontier API, {} the ${:.2f} open-API".format(
        util, vol, per_m, vs_front, API_FRONTIER, vs_open, API_OPEN))
print()
print("At tiny utilization the idle GPU loses to EVERYTHING; near saturation it crushes a frontier API but still")
print("loses to a cheap open-model API. Self-hosting is a bet on keeping the GPU BUSY.")
print("Self-host when volume is HIGH and STEADY, you need privacy/offline, or the open model is good enough.")
print("Stay on the API when volume is low or spiky (low util), you need frontier quality, or you have no ops capacity.")

# Output:
# True monthly TCO: $1095 GPU + $1500 ops (15 hrs x $100) = $2595/month.
# Raw GPU is only about 42% of the real cost - ops labor is the rest, and it never sleeps.
#
# Effective cost per 1M tokens depends entirely on UTILIZATION (actual volume vs the 6,500M capacity):
#   util  3% (  195M tok/mo):  $ 13.31/1M  ->  LOSES to the $10 frontier API, LOSES to the $0.40 open-API
#   util 20% ( 1300M tok/mo):  $  2.00/1M  ->  beats the $10 frontier API, LOSES to the $0.40 open-API
#   util 80% ( 5200M tok/mo):  $  0.50/1M  ->  beats the $10 frontier API, LOSES to the $0.40 open-API
#
# At tiny utilization the idle GPU loses to EVERYTHING; near saturation it crushes a frontier API but still
# loses to a cheap open-model API. Self-hosting is a bet on keeping the GPU BUSY.
# Self-host when volume is HIGH and STEADY, you need privacy/offline, or the open model is good enough.
# Stay on the API when volume is low or spiky (low util), you need frontier quality, or you have no ops capacity.

**Explanation**

The honest cost model that ties the lesson together: it adds ops labor to the GPU cost, then divides that total by the tokens actually served to get effective cost per million - which depends entirely on utilization. The point is that an idle GPU loses to every API, and even a saturated one usually still loses to a cheap open-model API, so self-hosting is a bet on keeping the GPU busy.

**How the code works - step by step**
- **`ops_cost` / `total_monthly`** - 15 hrs x $100/hr of engineer time added to the $1095 GPU, and the raw-GPU share of the true bill (~42%).
- **`CAPACITY_M`** - the tokens/month the GPU could serve if kept fully busy.
- **The utilization loop** - for 3%, 20%, and 80% utilization it computes actual volume, effective cost per 1M (`total_monthly / volume`), and whether that beats or loses to the frontier and open APIs.
- **Closing guidance** - self-host at high, steady volume (or for privacy/offline); stay on the API when volume is low or spiky, you need frontier quality, or you lack ops capacity.

**In one line:** effective cost = (GPU + ops) / tokens actually served - utilization, not the sticker price, decides.

**What you'll see:** True TCO is $2595/mo (GPU $1095 + ops $1500), with raw GPU only ~42% of it. At 3% utilization effective cost is $13.31/1M (loses to everything); at 20% it's $2.00 (beats frontier, loses to open); at 80% it's $0.50 (beats frontier, still loses to the $0.40 open API).

You now have the whole self-hosting decision as arithmetic: the crossover (Step 1) sets the break-even volume, Ollama runs a model in one command if it fits in memory (Step 2), quantization (Step 3) and the KV-cache budget (Step 4) are what make it fit, the OpenAI-compatible API plus a Modelfile (Step 5) wire it in with a one-line change, the serving knobs (Step 7) expose Ollama's one-at-a-time ceiling, and the real TCO (Step 8) shows that ops labor and utilization - not the sticker price - decide the outcome. The two questions to ask of any self-host bet: does your volume clear the break-even, and can you keep the GPU busy? When you need to serve many concurrent users, the one-at-a-time ceiling is the bottleneck - Lesson 13.6 graduates you to vLLM with continuous batching and PagedAttention.